# Customer Churn - step-by-step walkthrough
This notebook runs each stage of the workflow separately so you can inspect the output in between. The same code runs in one go with `python run_pipeline.py`.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from churn import config
from churn.acquire import acquire
from churn.clean import clean
from churn.validate import validate, profile, assert_valid
from churn.features import build_ml_ready
from churn.train import run_experiments, feature_importance
from churn.visualize import eda_figures

## 1. Acquire
Download the IBM Telco CSV, or fall back to the seeded generator.

In [ ]:
raw, source = acquire(prefer_url=True)
print(source, raw.shape)
raw.head()

## 2. Validate the raw data (expect failures)

In [ ]:
v_raw = validate(raw)
v_raw[~v_raw['passed']]

## 3. Clean

In [ ]:
df, log = clean(raw)
log

## 4. Validate again (all critical rules should pass)

In [ ]:
v = validate(df)
assert_valid(v)
print(f"{v['passed'].sum()}/{len(v)} rules passed")
profile(df)

## 5. EDA

In [ ]:
from IPython.display import Image, display
figs = eda_figures(df)
for p in figs.values():
    display(Image(filename=str(p)))

## 6. Feature engineering -> ML-ready table

In [ ]:
ml = build_ml_ready(df)
print(ml.shape)
ml.head()

## 7. Experiments

In [ ]:
exp = run_experiments(df)
exp['table'][['name','cv_roc_auc','roc_auc','precision','recall','f1']]

In [ ]:
feature_importance(exp['best'])

## 8. Conclusions
Write your own observations here after running the cells.